# Tới

# Dự án: Echo Valley - Dự án Phân tích Thương mại điện tử Olist
**Tên Notebook:** 04_eda_visualization.ipynb  
**Mục tiêu:** Thực hiện EDA (Exploratory Data Analysis) và trực quan hóa dữ liệu để tìm ra các điểm nghẽn (bottlenecks) trong chuỗi cung ứng và hành vi khách hàng.

# 1. Nạp dữ liệu sạch

## 1.1 Load dữ liệu đã làm sạch

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from math import radians, cos, sin, asin, sqrt
import warnings

warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    processed_dir = os.path.abspath(os.path.join(cwd, '..', 'data', 'processed'))
else:
    processed_dir = os.path.abspath(os.path.join(cwd, 'data', 'processed'))

os.makedirs(processed_dir, exist_ok=True)
processed_path = os.path.join(processed_dir, 'cleaned_data.parquet')
csv_path = os.path.join(processed_dir, 'cleaned_data.csv')

if os.path.exists(processed_path):
    df = pd.read_parquet(processed_path)
    print(f'Loaded cleaned data from Parquet: {processed_path}')
elif os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f'Loaded cleaned data from CSV: {csv_path}')
else:
    raise FileNotFoundError(f'Không tìm thấy dữ liệu đã xử lý. Vui lòng chạy Notebook 03 trước. Checked paths: {processed_path}, {csv_path}')

print(f'   Shape: {df.shape}')
print(f'   Columns: {list(df.columns)[:15]}...')


**Nhận xét:**
- Dữ liệu đã nạp thành công từ file cleaned_data.csv
- Kiểm tra shape và columns để xác nhận dữ liệu đầy đủ
- Sẵn sàng cho các bước phân tích tiếp theo (distance, delivery time, visualization)

# 2. Phân tích địa lý và Khoảng cách

## 2.1 Tính toán khoảng cách (Distance Engineering)

### 2.1.1 Tính khoảng cách Haversine giữa khách hàng và người bán

**a.** Tính khoảng cách địa lý giữa tọa độ khách hàng và người bán

In [ ]:
def haversine_distance_vectorized(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371 * c

print('Tính toán khoảng cách địa lý (Haversine)...')

# Find data/archive path
cwd = os.getcwd()
archive_candidates = [
    os.path.join(cwd, 'data', 'archive'),
    os.path.join(cwd, '..', 'data', 'archive')
]
archive_dir = None
for cand in archive_candidates:
    if os.path.exists(cand):
        archive_dir = os.path.abspath(cand)
        break

if not archive_dir:
    raise FileNotFoundError("Không tìm thấy thư mục data/archive.")

geo_path = os.path.join(archive_dir, 'olist_geolocation_dataset.csv')
if os.path.exists(geo_path):
    print("Đang đọc geolocation dataset...")
    geo = pd.read_csv(geo_path)
    print("Tính toán trung vị tọa độ theo zipcode...")
    geo_avg = geo.groupby('geolocation_zip_code_prefix', as_index=False).agg({
        'geolocation_lat': 'mean',
        'geolocation_lng': 'mean'
    })

    # Remove duplicates or coordinates from previous merges if any
    drop_cols = [c for c in ['customer_lat', 'customer_lon', 'seller_lat', 'seller_lon'] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    # Merge customer coordinates
    if 'customer_zip_code_prefix' in df.columns:
        df = df.merge(
            geo_avg.rename(columns={
                'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
                'geolocation_lat': 'customer_lat',
                'geolocation_lng': 'customer_lon'
            })[['customer_zip_code_prefix', 'customer_lat', 'customer_lon']],
            on='customer_zip_code_prefix',
            how='left'
        )
        print('Đã ghép tọa độ khách hàng từ geolocation dataset.')

    # Merge seller coordinates
    if 'seller_zip_code_prefix' in df.columns:
        df = df.merge(
            geo_avg.rename(columns={
                'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
                'geolocation_lat': 'seller_lat',
                'geolocation_lng': 'seller_lon'
            })[['seller_zip_code_prefix', 'seller_lat', 'seller_lon']],
            on='seller_zip_code_prefix',
            how='left'
        )
        print('Đã ghép tọa độ người bán từ geolocation dataset.')
else:
    print(f'Không tìm thấy file geolocation để ghép tọa độ: {geo_path}')

required_cols = ['customer_lat', 'customer_lon', 'seller_lat', 'seller_lon']
if all(col in df.columns for col in required_cols):
    df['distance_km'] = haversine_distance_vectorized(
        df['customer_lat'], df['customer_lon'],
        df['seller_lat'], df['seller_lon']
    )
    non_null_dist = df['distance_km'].dropna()
    if not non_null_dist.empty:
        print(f"Distance calculated! Min={non_null_dist.min():.2f}, Max={non_null_dist.max():.2f}, Mean={non_null_dist.mean():.2f}")
    else:
        print("Tất cả khoảng cách đều là NaN (không ghép được tọa độ phù hợp).")
else:
    missing_cols = [col for col in required_cols if col not in df.columns]
    print('Không tìm thấy đủ cột tọa độ để tính distance:')
    print(missing_cols)


**Nhận xét:**
- Khoảng cách đã được tính toán bằng Haversine formula từ tọa độ GPS
- Khoảng cách dao động từ 0 km đến hàng ngàn km (tùy theo vị trí khách hàng và người bán)
- Có khả năng cao là khoảng cách dài → thời gian giao hàng tăng (correlation dương)
- Các đơn hàng xuyên quốc gia sẽ là bottleneck lớn nhất

# 3. Phân tích điểm nghẽn (Bottleneck Analysis)

## 3.1 Thời gian giao hàng thực tế vs. Dự kiến

### 3.1.1 So sánh thời gian giao hàng

**a.** Tính hiệu số: thời gian giao thực tế vs thời gian ước tính

In [ ]:
print('Phân tích Thời gian Giao hàng...')

# Rename if misspelled
if 'order_deliverd_customer_date' in df.columns and 'order_delivered_customer_date' not in df.columns:
    df = df.rename(columns={'order_deliverd_customer_date': 'order_delivered_customer_date'})

required_cols = [
    'order_purchase_timestamp',
    'order_delivered_customer_date'
]

if all(col in df.columns for col in required_cols):
    df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'], errors='coerce')
    df['order_delivered_customer_date'] = pd.to_datetime(df['order_delivered_customer_date'], errors='coerce')
    
    df['actual_delivery_days'] = (df['order_delivered_customer_date'] - df['order_purchase_timestamp']).dt.days

    if 'order_estimated_delivery_date' in df.columns:
        df['order_estimated_delivery_date'] = pd.to_datetime(df['order_estimated_delivery_date'], errors='coerce')
        df['delivery_delay_days'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days

        non_null_delay = df['delivery_delay_days'].dropna()
        if not non_null_delay.empty:
            on_time_pct = (df['delivery_delay_days'] <= 0).mean() * 100
            delayed_pct = (df['delivery_delay_days'] > 0).mean() * 100

            print('Delivery Performance:')
            print(f'   On-time: {on_time_pct:.1f}%')
            print(f'   Delayed: {delayed_pct:.1f}%')
            print(f'   Avg delay: {df["delivery_delay_days"].mean():.1f} days')
        else:
            print('Không có dữ liệu trễ giao hàng.')
else:
    print('Thiếu cột cần thiết để tính thời gian giao hàng.')
    print('Các cột hiện có:')
    print(df.columns.tolist())


**Nhận xét:**
- Thời gian giao hàng trung bình: thường 5-30 ngày tùy khoảng cách
- Các đơn hàng bị chậm (delay > 0) là điểm cần cải thiện trong quy trình
- Chiến lược cải thiện:
  - Tăng số lượng seller/warehouse ở các khu vực xa
  - Tối ưu hóa tuyến đường giao hàng
  - Áp dụng logistics partner khác cho đơn hàng xa

# 4. Trực quan hóa xu hướng & Hành vi

## 4.1 Phân tích các yếu tố ảnh hưởng

### 4.1.1 Biểu đồ trực quan hóa (Heatmap, Bar chart, v.v.)

**a.** Vẽ biểu đồ phân tích dữ liệu quan trọng

In [ ]:
# Create numeric df for correlation heatmap
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

# Find path to save images
cwd = os.getcwd()
if os.path.basename(cwd) == 'notebooks':
    processed_dir = os.path.abspath(os.path.join(cwd, '..', 'data', 'processed'))
else:
    processed_dir = os.path.abspath(os.path.join(cwd, 'data', 'processed'))

os.makedirs(processed_dir, exist_ok=True)

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix - Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
heatmap_path = os.path.join(processed_dir, 'correlation_matrix.png')
plt.savefig(heatmap_path, dpi=100, bbox_inches='tight')
plt.show()
print(f'Saved correlation heatmap to: {heatmap_path}')

if 'order_status' in df.columns:
    plt.figure(figsize=(10, 6))
    status_counts = df['order_status'].value_counts(dropna=False)
    sns.barplot(x=status_counts.index, y=status_counts.values, palette='Blues_r')
    plt.title('Order Status Distribution', fontsize=14, fontweight='bold')
    plt.xlabel('Status')
    plt.ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    status_path = os.path.join(processed_dir, 'order_status.png')
    plt.savefig(status_path, dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Saved order status distribution to: {status_path}')
else:
    print('Không tìm thấy cột order_status để hiển thị distribution.')


**Nhận xét:**
- Heatmap Correlation:
  - Các biến có tương quan cao → có thể dùng cho feature engineering
  - Detect multicollinearity trước khi training ML model
- Order Status Distribution:
  - Kiểm tra tỷ lệ delivered vs cancelled/problematic orders
  - Nếu cancelled > 10%, cần điều tra nguyên nhân chính
- Key Findings: Xác định top 3 bottleneck để tập trung cải thiện